In [1]:
import torch
import torchvision
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms
from torch.autograd import Variable
import torchvision.datasets as d_sets
from torch.utils.data import DataLoader as d_loader
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn.functional as F
import os
import os.path as osp
import logging
from collections import OrderedDict
import json
from datetime import datetime
import os
import math
import numpy as np
from torchvision.utils import make_grid
import numpy as np
import torch
from torch.utils.data import Dataset

import logging
from re import split


In [ ]:
!nvidia-smi

In [ ]:
import torch


print("Available GPUs count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print("GPU", i, ":", torch.cuda.get_device_name(i))

In [4]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()

In [5]:
import torch
torch.cuda.set_device(1)

In [6]:
def tensor2numpy(tensor, mean, std, mean_input_wind_lr, std_input_wind_lr):
    '''
    Converts a torch Tensor into a numpy array and de-normalizes it using given mean and std
    Input: 3D(1,H,W) or 4D(1,1,H,W)
    Output: 3D(1,H,W) or 4D(1,1,H,W), float32
    '''
    if tensor.dim() == 3 and tensor.size(0) == 1:
        tensor = tensor.squeeze(0)  # Remove the first dimension (C, H, W)
        tensor = tensor.float().cpu()
        
        # De-normalize using the given mean and std
        tensor = tensor * std + mean
        
        img_np = tensor.numpy()
        
        
    
    elif tensor.dim() == 4 and tensor.size(0) == 1 and tensor.size(1) == 1:
        tensor = tensor.squeeze(0).squeeze(0)  # Remove the first two dimensions (H, W)
        tensor = tensor.float().cpu()
        
        # De-normalize using the given mean and std
        tensor = tensor * std + mean
        
        img_np = tensor.numpy()
        
    elif tensor.dim() == 4 and tensor.size(0) == 1 and tensor.size(1) == 2:
        # Extract the third channel
        tensor = tensor.squeeze(0)  # Remove the first dimension (3, H, W)
        tensor = tensor[1, :, :]  # Extract the third channel (H, W)
        tensor = tensor.float().cpu()
        
        # De-normalize using the given mean_input_wind_lr and std_input_wind_lr
        tensor = tensor * std_input_wind_lr + mean_input_wind_lr
        
        img_np = tensor.numpy()
        
       
    
    else:
        raise TypeError('Only support 3D tensor with first dimension as 1 or 4D tensor with first two dimensions as 1. But received with dimension: {:d} and size: {}'.format(tensor.dim(), tensor.size()))
    
    return img_np.astype(np.float32)

def save_img(img, img_path):

    np.save(img_path, img)


def calculate_psnr(img1, img2):
    # img1 and img2 have range [0, 255]
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse = np.mean((img1 - img2)**2)
    if mse == 0:
        return float('inf')
    max_value = max(np.max(img1), np.max(img2))
    return 20 * math.log10(max_value / math.sqrt(mse))

def calculate_mse(img1, img2):
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse = np.mean((img1 - img2)**2)
    return mse


In [7]:
import numpy as np
import torch
from torch.utils.data import Dataset

class LRHR_dataset(Dataset):
    def __init__(self, input_file, output_file):


        self.input_data = np.load(input_file)
        self.output_data = np.load(output_file)

  
        self.normalized_input = np.zeros_like(self.input_data)
        for i in range(self.input_data.shape[1]): 
            mean = np.mean(self.input_data[:, i, :, :], axis=0, keepdims=True)
            std = np.std(self.input_data[:, i, :, :], axis=0, keepdims=True)
            if np.any(std == 0): 
                print(f"Variable {i} has at least one std=0, skipping normalization for this variable.")
                self.normalized_input[:, i, :, :] = self.input_data[:, i, :, :]
            else:
                self.normalized_input[:, i, :, :] = (self.input_data[:, i, :, :] - mean) / std
            if i == 1:
                self.mean_i2 = mean
                self.std_i2 = std

        self.mean_output = np.mean(self.output_data, axis=0, keepdims=True)
        self.std_output = np.std(self.output_data, axis=0, keepdims=True)
        self.normalized_output = (self.output_data - self.mean_output) / self.std_output

    def __len__(self):

        return self.input_data.shape[0]

    def __getitem__(self, index):

        input_sample = self.normalized_input[index]
        output_sample = self.normalized_output[index]


        input_tensor = torch.from_numpy(input_sample).float()
        output_tensor = torch.from_numpy(output_sample[np.newaxis, :, :]).float()

        return input_tensor, output_tensor
    
    def get_output_mean_std(self):

        return self.mean_output, self.std_output
    def get_input_i2_mean_std(self):

        return self.mean_i2, self.std_i2



In [8]:
import torch
import torchvision
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms
from torch.autograd import Variable
import torchvision.datasets as d_sets
from torch.utils.data import DataLoader as d_loader
import matplotlib.pyplot as plt
from PIL import Image
import torch.nn.functional as F


import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torch


class conv_block(nn.Module):
    """
    Convolution Block 
    """
    def __init__(self, in_ch, out_ch):
        super(conv_block, self).__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True))

    def forward(self, x):

        x = self.conv(x)
        return x


class up_conv(nn.Module):
    """
    Up Convolution Block
    """
    def __init__(self, in_ch, out_ch):
        super(up_conv, self).__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.up(x)
        return x





class Recurrent_block(nn.Module):
    """
    Recurrent Block for R2Unet_CNN
    """
    def __init__(self, out_ch, t=2):
        super(Recurrent_block, self).__init__()

        self.t = t
        self.out_ch = out_ch
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        for i in range(self.t):
            if i == 0:
                x = self.conv(x)
            out = self.conv(x + x)
        return out


class RRCNN_block(nn.Module):
    """
    Recurrent Residual Convolutional Neural Network Block
    """
    def __init__(self, in_ch, out_ch, t=2):
        super(RRCNN_block, self).__init__()

        self.RCNN = nn.Sequential(
            Recurrent_block(out_ch, t=t),
            Recurrent_block(out_ch, t=t)
        )
        self.Conv = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x1 = self.Conv(x)
        x2 = self.RCNN(x1)
        out = x1 + x2
        return out



class Attention_block(nn.Module):
    """
    Attention Block
    """

    def __init__(self, F_g, F_l, F_int):
        super(Attention_block, self).__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        out = x * psi
        return out


class AttU_Net(nn.Module):
    """
    Attention Unet implementation
    Paper: https://arxiv.org/abs/1804.03999
    """
    def __init__(self, img_ch=2, output_ch=1):
        super(AttU_Net, self).__init__()

        n1 = 128
        filters = [n1, n1 * 2, n1 * 4, n1 * 8, n1 * 16]

        self.Maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.Maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.Maxpool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.Maxpool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.Conv1 = conv_block(img_ch, filters[0])
        self.Conv2 = conv_block(filters[0], filters[1])
        self.Conv3 = conv_block(filters[1], filters[2])
        self.Conv4 = conv_block(filters[2], filters[3])
        self.Conv5 = conv_block(filters[3], filters[4])

        self.Up5 = up_conv(filters[4], filters[3])
        self.Att5 = Attention_block(F_g=filters[3], F_l=filters[3], F_int=filters[2])
        self.Up_conv5 = conv_block(filters[4], filters[3])

        self.Up4 = up_conv(filters[3], filters[2])
        self.Att4 = Attention_block(F_g=filters[2], F_l=filters[2], F_int=filters[1])
        self.Up_conv4 = conv_block(filters[3], filters[2])

        self.Up3 = up_conv(filters[2], filters[1])
        self.Att3 = Attention_block(F_g=filters[1], F_l=filters[1], F_int=filters[0])
        self.Up_conv3 = conv_block(filters[2], filters[1])

        self.Up2 = up_conv(filters[1], filters[0])
        self.Att2 = Attention_block(F_g=filters[0], F_l=filters[0], F_int=32)
        self.Up_conv2 = conv_block(filters[1], filters[0])

        self.Conv = nn.Conv2d(filters[0], output_ch, kernel_size=1, stride=1, padding=0)

        #self.active = torch.nn.Sigmoid()


    def forward(self, x):

        e1 = self.Conv1(x)

        e2 = self.Maxpool1(e1)
        e2 = self.Conv2(e2)

        e3 = self.Maxpool2(e2)
        e3 = self.Conv3(e3)

        e4 = self.Maxpool3(e3)
        e4 = self.Conv4(e4)

        e5 = self.Maxpool4(e4)
        e5 = self.Conv5(e5)

        #print(x5.shape)
        d5 = self.Up5(e5)
        #print(d5.shape)
        x4 = self.Att5(g=d5, x=e4)
        d5 = torch.cat((x4, d5), dim=1)
        d5 = self.Up_conv5(d5)

        d4 = self.Up4(d5)
        x3 = self.Att4(g=d4, x=e3)
        d4 = torch.cat((x3, d4), dim=1)
        d4 = self.Up_conv4(d4)

        d3 = self.Up3(d4)
        x2 = self.Att3(g=d3, x=e2)
        d3 = torch.cat((x2, d3), dim=1)
        d3 = self.Up_conv3(d3)

        d2 = self.Up2(d3)
        x1 = self.Att2(g=d2, x=e1)
        d2 = torch.cat((x1, d2), dim=1)
        d2 = self.Up_conv2(d2)

        out = self.Conv(d2)

      #  out = self.active(out)

        return out



In [ ]:
class Args:
    def __init__(self, batch_size=16, test_batch_size=1, epochs=400, lr=0.0003, cuda=1, threads=16, seed=123,large_kernel_size = 9,small_kernel_size = 3,n_channels = 32,n_blocks = 16):
        
        self.batch_size = batch_size
        self.test_batch_size = test_batch_size
        self.epochs = epochs
        self.lr = lr
        self.cuda = cuda
        self.threads = threads
        self.seed = seed
        self.large_kernel_size =  large_kernel_size 
        self.small_kernel_size = small_kernel_size  
        self.n_channels = n_channels       
        self.n_blocks = n_blocks         
        #self.scaling_factor = scaling_factor
opt = Args()

print(opt)
print(opt.batch_size)
print(opt.n_channels)

In [ ]:


import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torch.utils.data import DataLoader, random_split

use_cuda = opt.cuda
if use_cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(opt.seed)
if use_cuda:
    torch.cuda.set_device(1)
    torch.cuda.manual_seed(opt.seed)
    
input_file_trainset = ""
output_file_trainset = ""
input_file_testset = ""
output_file_testset = ""

inference_input_file = ''
inference_output_fake_random_file= ''


train_dataset = LRHR_dataset(input_file_trainset,output_file_trainset)
test_dataset = LRHR_dataset(input_file_testset,output_file_testset)

inference_dataset = LRHR_dataset(inference_input_file,inference_output_fake_random_file)

train_mean_output,train_std_output = train_dataset.get_output_mean_std()
train_mean_input, train_std_input = train_dataset.get_input_i2_mean_std()
print(train_std_output)
test_mean_output, test_std_output = test_dataset.get_output_mean_std()
test_mean_input, test_std_input = test_dataset.get_input_i2_mean_std()

inference_mean_input, inference_std_input = inference_dataset.get_input_i2_mean_std()
print(inference_mean_input)



training_data_loader = DataLoader(dataset=train_dataset, num_workers=opt.threads, batch_size=opt.batch_size, shuffle=True)
for batch_datas, batch_labels in training_data_loader:
    print(batch_datas.size(),batch_labels.size())
testing_data_loader = DataLoader(dataset=test_dataset, num_workers=opt.threads, batch_size=opt.test_batch_size, shuffle=False)

inference_data_loader = DataLoader(dataset=inference_dataset, num_workers=opt.threads, batch_size=1, shuffle=False)
for batch_datas, batch_labels in inference_data_loader:
    print(batch_datas.size(),batch_labels.size())

In [11]:
import logging


my_logger = logging.Logger("first_logger")


my_handler = logging.FileHandler('test_only2var_lr_0.0003.log')


my_handler.setLevel(logging.INFO)
my_format = logging.Formatter("时间:%(asctime)s 日志信息:%(message)s 行号:%(lineno)d")


my_handler.setFormatter(my_format)
my_logger.addHandler(my_handler)




In [ ]:
  

def load_model_from_pth(file_path):
    model = torch.load(file_path, map_location={'cuda:1':'cuda:1'} )
    #model = torch.load(file_path )

    print(next(model.parameters()).device)

    return model

def inference(pth_file):
    #avg_psnr = 0
    
    srcnn = load_model_from_pth(pth_file)
    srcnn.eval()
    
    current_step = 0
    with torch.no_grad():
        for batch in inference_data_loader:
            current_step +=1
            input = Variable(batch[0])
            if use_cuda:
                input = input.cuda()
                

            prediction = srcnn(input)
            
            
            
            sr_image = prediction.detach().cpu()
            
            lr_image = input.detach().cpu()
            #print("sr shape:",sr_image.shape)
            #print("hr shape:",hr_image.shape)
            #print("lr shape:",lr_image.shape)
            
            
            sr_img = tensor2numpy(sr_image,train_mean_output,train_std_output,inference_mean_input, inference_std_input) # uint8
            
            lr_img = tensor2numpy(lr_image,train_mean_output,train_std_output,inference_mean_input, inference_std_input) # uint8
            
            inference_path = '/data/hanzhe/era5_to_5km_inference/fgoals-g3/fgoals-g3_inference_historical_results'
            
            result_dir = os.path.join(inference_path, 'inference_results')
            os.makedirs(result_dir, exist_ok=True)
            
            
       
            
            save_img(sr_img, os.path.join(result_dir, f'{current_step}_sr.npy'))
            save_img(lr_img, os.path.join(result_dir, f'{current_step}_lr.npy'))
            
            
            my_logger.info(f'Prediction for step {current_step} saved at {result_dir}')
            

            
pth_file = '/data/hanzhe/era5_to_5km_inference/model_epoch_193.pth'  
inference(pth_file)